# Minimalna moc w układzie mostkowym - programowanie liniowe

**Cel ćwiczenia:** wyznaczyć prądy w gałęziach mostka tak, aby całkowita moc rozpraszana na rezystorach była najmniejsza. Napięcia na rezystorach są znane, a prądy mogą zmieniać się w podanych przedziałach.

| Rezystor | Napięcie [V] | Prąd nominalny [mA] | Tolerancja [mA] |
|:---:|---:|---:|---:|
| $R_1$ | 6 | 4 | 1 |
| $R_2$ | 10 | 2 | 1 |
| $R_3$ | 4 | 2 | 1 |
| $R_4$ | 7 | 2 | 1 |
| $R_5$ | 3 | 4 | 1 |

Zakładamy kierunki prądów zgodne ze schematem. Rezystancje nie są podane, dlatego traktujemy je jako wartości, które można dobrać do wyznaczonych prądów.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle, FancyArrowPatch

U = np.array([6, 10, 4, 7, 3], dtype=float)
I_nom = np.array([4, 2, 2, 2, 4], dtype=float)
delta = np.ones(5)

def rysuj_schemat(I=None):
    fig, ax = plt.subplots(figsize=(9, 5))
    A, B, C, D = [np.array(p, dtype=float) for p in [(0,0),(3,2),(3,-2),(6,0)]]
    galezie = [(A,B),(A,C),(B,C),(B,D),(C,D)]
    przesuniecia = [(-0.45,0.60),(-0.45,-0.65),(0.72,0),(0.45,0.60),(0.45,-0.65)]
    for k, (a,b) in enumerate(galezie):
        e = (b-a)/np.linalg.norm(b-a)
        f = np.array([-e[1],e[0]])
        s = (a+b)/2
        l, h = 0.48, 0.16
        ax.plot(*np.array([a,s-l*e]).T, color='black', lw=1.7)
        ax.plot(*np.array([s+l*e,b]).T, color='black', lw=1.7)
        ax.add_patch(Polygon([s-l*e-h*f,s+l*e-h*f,s+l*e+h*f,s-l*e+h*f],
                             closed=True, facecolor='white', edgecolor='black', lw=1.7))
        ax.add_patch(FancyArrowPatch(a+0.13*(b-a),a+0.30*(b-a),arrowstyle='-|>',
                                    mutation_scale=13,color='tab:blue',lw=1.5))
        t = s + przesuniecia[k]
        etykieta = f'$R_{k+1}$, {U[k]:g} V'
        if I is not None:
            etykieta += f'\n$I_{k+1}$ = {I[k]:g} mA'
        else:
            etykieta += f'\n$I_{k+1}$'
        ax.text(*t, etykieta, ha='center', va='center', fontsize=11)
    ax.plot([-0.9,0],[0,0],color='black',lw=1.7)
    ax.plot([6,6.9],[0,0],color='black',lw=1.7)
    for punkt in (A,B,C,D):
        ax.add_patch(Circle(punkt,0.055,color='black'))
    for punkt, nazwa, offset in [(A,'A',(-0.15,0.25)),(B,'B',(0,0.25)),(C,'C',(0,-0.28)),(D,'D',(0.15,0.25))]:
        ax.text(*(punkt+offset),nazwa,ha='center',fontsize=11)
    ax.set_aspect('equal')
    ax.set_xlim(-1.2,7.2)
    ax.set_ylim(-2.8,2.8)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

rysuj_schemat()

/tmp/ipykernel_1383/1421361162.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1. Wybór zmiennych decyzyjnych

Zmienne decyzyjne to prądy w pięciu rezystorach, wyrażone w mA:

$$\mathbf{x}=[I_1,I_2,I_3,I_4,I_5]^T.$$

Moc na każdym rezystorze obliczamy ze wzoru $P_i=U_i I_i$. Ponieważ napięcia są stałe, funkcja celu jest liniowa:

$$\min P=6I_1+10I_2+4I_3+7I_4+3I_5.$$

Prąd podany w mA i napięcie w V dają moc w mW.

### Pierwsze prawo Kirchhoffa

Dla górnego węzła B:

$$I_1=I_3+I_4.$$

Dla dolnego węzła C:

$$I_2+I_3=I_5.$$

### Drugie prawo Kirchhoffa

Sprawdzamy podane napięcia w obu oczkach:

$$U_1+U_3-U_2=6+4-10=0,$$
$$U_3+U_5-U_4=4+3-7=0.$$

Napięcia spełniają drugie prawo Kirchhoffa. Nie trzeba dodawać kolejnych równań prądowych wynikających z tych spadków, ponieważ są one już ustalone.

### Ograniczenia prądów

Dla każdego rezystora obowiązuje przedział $I_{i,nom}-\Delta_i\leq I_i\leq I_{i,nom}+\Delta_i$.

$$3\leq I_1\leq5,\quad 1\leq I_2\leq3,\quad 1\leq I_3\leq3,$$
$$1\leq I_4\leq3,\quad 3\leq I_5\leq5.$$

## 2. Macierzowa postać zagadnienia

Funkcję celu zapisujemy w postaci $\min \mathbf{c}^T\mathbf{x}$, gdzie:

$$\mathbf{c}=[6,10,4,7,3]^T.$$

Równania węzłowe w postaci macierzowej:

$$\begin{bmatrix}1&0&-1&-1&0\\0&1&1&0&-1\end{bmatrix}\begin{bmatrix}I_1\\I_2\\I_3\\I_4\\I_5\end{bmatrix}=\begin{bmatrix}0\\0\end{bmatrix}.$$

Ograniczenia przedziałowe przekazujemy do solvera jako `bounds`.

In [2]:
import pandas as pd
from scipy.optimize import linprog

nazwy = ['R1', 'R2', 'R3', 'R4', 'R5']

# Wspolczynniki funkcji celu
c = U.copy()

# Rownania pradowe dla wezlow B i C
A_eq = np.array([
    [1, 0, -1, -1, 0],
    [0, 1, 1, 0, -1]
])
b_eq = np.zeros(2)

# Dolne i gorne granice pradow
bounds = list(zip(I_nom - delta, I_nom + delta))

print('Macierz A_eq:')
print(A_eq)
print('Wektor b_eq:', b_eq)
print('Wektor c:', c)
print('Ograniczenia:', bounds)

Macierz A_eq:
[[ 1  0 -1 -1  0]
 [ 0  1  1  0 -1]]
Wektor b_eq: [0. 0.]
Wektor c: [ 6. 10.  4.  7.  3.]
Ograniczenia: [(np.float64(3.0), np.float64(5.0)), (np.float64(1.0), np.float64(3.0)), (np.float64(1.0), np.float64(3.0)), (np.float64(1.0), np.float64(3.0)), (np.float64(3.0), np.float64(5.0))]


## 3. Obliczenie optimum za pomocą solvera

Korzystamy z funkcji `linprog`, tak samo jak w poprzednim zadaniu. Tym razem nie zmieniamy znaku współczynników funkcji celu, ponieważ szukamy minimum.

In [3]:
wynik = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
if not wynik.success:
    raise RuntimeError(wynik.message)

I = wynik.x
P = U * I
zestawienie = pd.DataFrame({
    'Rezystor': nazwy,
    'U [V]': U,
    'I [mA]': I,
    'P [mW]': P,
    'R [kOhm]': U / I
})
print(zestawienie.to_string(index=False))
print(f'\nMinimalna moc: {wynik.fun:g} mW')
print(f'Prad calkowity: {I[0] + I[1]:g} mA')
print('Bilans pradow w wezlach:', A_eq @ I)

Rezystor  U [V]  I [mA]  P [mW]  R [kOhm]
      R1    6.0     3.0    18.0       2.0
      R2   10.0     1.0    10.0      10.0
      R3    4.0     2.0     8.0       2.0
      R4    7.0     1.0     7.0       7.0
      R5    3.0     3.0     9.0       1.0

Minimalna moc: 52 mW
Prad calkowity: 4 mA
Bilans pradow w wezlach: [0. 0.]


### Rozkład prądów przy minimalnej mocy

In [4]:
rysuj_schemat(I)

/tmp/ipykernel_1383/1421361162.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Sprawdzenie wyniku bez solvera

Z równań węzłowych mamy $I_3=I_1-I_4$ oraz $I_5=I_2+I_1-I_4$. Po podstawieniu do funkcji celu:

$$P=6I_1+10I_2+4(I_1-I_4)+7I_4+3(I_2+I_1-I_4)$$

$$P=13(I_1+I_2).$$

Najmniejsze dopuszczalne wartości prądów wejściowych to $I_1=3$ mA i $I_2=1$ mA. Dla tych wartości z warunków $I_4\geq1$ i $I_5\geq3$ wynika jednocześnie $I_3\leq2$ oraz $I_3\geq2$. Zatem $I_3=2$ mA, $I_4=1$ mA i $I_5=3$ mA.

Wszystkie ograniczenia są spełnione, więc otrzymujemy minimum:

$$P_{min}=13\cdot(3+1)=52\ \mathrm{mW}=0{,}052\ \mathrm{W}.$$

## 5. Podsumowanie

Minimalna moc rozpraszana na mostku wynosi 52 mW. Otrzymane prądy mieszczą się w zadanych przedziałach i spełniają prawa Kirchhoffa. Programowanie liniowe pozwoliło znaleźć rozwiązanie bez wcześniejszego wyznaczania rezystancji zastępczej. Z podanych napięć i obliczonych prądów można następnie wyznaczyć rezystancje poszczególnych elementów.